# COT_lab — Results

Loads `outputs/eval_results/` and renders the final accuracy table, ReCEval summary, and per-condition plots.
Run after Stage 5 has produced its CSVs.

**Conditions**

| Condition | Model | Train set | Notes |
|---|---|---|---|
| `baseline` | FLAN-T5-base | — | Zero-shot |
| `student_direct_ft` | FLAN-T5-base | Direct FT | Answer-only targets |
| `student_set_a` | FLAN-T5-base | Set A — all CoTs | ~7 473 examples incl. wrong-answer chains |
| `student_set_b` | FLAN-T5-base | Set B — Magister filter | Correct-answer CoTs only (~3 389) |
| `student_set_c` | FLAN-T5-base | Set C — calculator-patched | Set B with corrected intermediate arithmetic |
| `student_set_c_mix` | FLAN-T5-base | Set C + Direct FT | Mixed CoT + answer-only objective |
| `student_direct_ft_large` | FLAN-T5-large | Direct FT | Scale crossover |
| `student_set_b_large` | FLAN-T5-large | Set B | Scale crossover |
| `student_set_c_large` | FLAN-T5-large | Set C | Scale crossover |

**Reference** (Ho et al. 2023, FLAN-T5-base on GSM8K): baseline 2.50 %, standard FT 5.08 %, CoT FT 4.40 %.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_ROOT = Path('..').resolve()
EVAL  = REPO_ROOT / 'outputs' / 'eval_results'
RUNS  = REPO_ROOT / 'outputs' / 'runs'
PLOTS = REPO_ROOT / 'outputs' / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

ORDER = [
    'baseline',
    'student_direct_ft',
    'student_set_a',
    'student_set_b',
    'student_set_c',
    'student_set_c_mix',
    'student_direct_ft_large',
    'student_set_b_large',
    'student_set_c_large',
]

def _sort(df):
    df['_o'] = df['condition'].map({c: i for i, c in enumerate(ORDER)}).fillna(99)
    return df.sort_values('_o').drop(columns='_o').reset_index(drop=True)

def _colors(conditions):
    return ['darkorange' if 'large' in c else 'steelblue' for c in conditions]

print('eval dir:', EVAL)
print('files:   ', sorted(p.name for p in EVAL.iterdir()) if EVAL.exists() else 'NOT FOUND')

## 1. Accuracy

Reference (Ho et al. 2023, FLAN-T5-base on GSM8K): baseline 2.50 %, standard FT 5.08 %, CoT FT 4.40 %.

In [ ]:
acc = _sort(pd.read_csv(EVAL / 'accuracy.csv'))
acc['accuracy_pct'] = (acc['accuracy'] * 100).round(2)
acc[['condition', 'n', 'correct', 'accuracy_pct']]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(acc['condition'], acc['accuracy_pct'], color=_colors(acc['condition']))
ax.set_xticklabels(acc['condition'], rotation=30, ha='right')
ax.set_ylabel('accuracy (%)')
ax.set_title('GSM8K test accuracy by condition')
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, acc['accuracy_pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.04,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=8)

l1 = ax.axhline(2.50, color='grey', linestyle=':',  linewidth=0.9, label='Ho et al. baseline (2.50%)')
l2 = ax.axhline(5.08, color='grey', linestyle='--', linewidth=0.9, label='Ho et al. direct FT (5.08%)')
l3 = ax.axhline(4.40, color='grey', linestyle='-.', linewidth=0.9, label='Ho et al. CoT FT (4.40%)')

ax.legend(handles=[
    mpatches.Patch(color='steelblue',  label='FLAN-T5-base'),
    mpatches.Patch(color='darkorange', label='FLAN-T5-large'),
    l1, l2, l3,
], fontsize=7, loc='upper left')

fig.tight_layout()
fig.savefig(PLOTS / 'accuracy_bar.png', dpi=150)
plt.show()

## 2. ReCEval summary

Metrics — **intra**: step-level coherence (NLI entailment, higher = more self-consistent steps). **inter**: inter-step relevance (contradiction score vs. prior context, higher = less contradiction). **info**: informativeness (log-perplexity delta vs. distilgpt2, positive = step helps predict the gold answer).

> Per-example `.jsonl` files are not committed (too large). This section uses the pre-aggregated `receval_summary.csv`.

In [ ]:
rec = _sort(pd.read_csv(EVAL / 'receval_summary.csv'))
rec[['condition', 'n',
     'intra_mean', 'intra_std',
     'inter_mean', 'inter_std',
     'info_mean',  'info_std']].round(4)

In [ ]:
METRIC_META = [
    ('intra', 'Intra-step coherence'),
    ('inter', 'Inter-step relevance'),
    ('info',  'Informativeness'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (m, title) in zip(axes, METRIC_META):
    ax.bar(range(len(rec)), rec[f'{m}_mean'], yerr=rec[f'{m}_std'],
           color=_colors(rec['condition']), capsize=3, error_kw={'linewidth': 0.8})
    ax.set_xticks(range(len(rec)))
    ax.set_xticklabels(rec['condition'], rotation=30, ha='right', fontsize=7)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

fig.legend(handles=[
    mpatches.Patch(color='steelblue',  label='FLAN-T5-base'),
    mpatches.Patch(color='darkorange', label='FLAN-T5-large'),
], loc='upper right', fontsize=8)
fig.suptitle('ReCEval means ± 1 std by condition')
fig.tight_layout()
fig.savefig(PLOTS / 'receval_bar.png', dpi=150)
plt.show()

## 3. Accuracy × ReCEval scatter

Each point is one condition. Tests whether higher GSM8K accuracy correlates with better reasoning quality scores.

In [ ]:
merged = acc[['condition', 'accuracy_pct']].merge(
    rec[['condition', 'intra_mean', 'inter_mean', 'info_mean']], on='condition'
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (m, title) in zip(axes, METRIC_META):
    ax.scatter(merged['accuracy_pct'], merged[f'{m}_mean'],
               c=_colors(merged['condition']), s=70, zorder=3)
    for _, row in merged.iterrows():
        label = row['condition'].replace('student_', '').replace('_large', ' (L)')
        ax.annotate(label, (row['accuracy_pct'], row[f'{m}_mean']),
                    fontsize=6, xytext=(4, 2), textcoords='offset points')
    ax.set_xlabel('accuracy (%)')
    ax.set_ylabel(m)
    ax.set_title(f'accuracy vs {title}')
    ax.grid(alpha=0.3)

fig.legend(handles=[
    mpatches.Patch(color='steelblue',  label='FLAN-T5-base'),
    mpatches.Patch(color='darkorange', label='FLAN-T5-large'),
], loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS / 'accuracy_vs_receval.png', dpi=150)
plt.show()

## 4. Run-card summary

Training / inference / ReCEval metadata from `outputs/runs/`.

In [ ]:
rows = []
for cond in ORDER:
    row = {'condition': cond}
    for stage, prefix in [('train', '03_train'), ('infer', '04_inference'), ('receval', '05b')]:
        card = RUNS / f'{prefix}_{cond}.json'
        if card.exists():
            d = json.loads(card.read_text())
            row[f'{stage}_status'] = d.get('status', '?')
            row[f'{stage}_dur_s']  = round(d.get('duration_seconds', float('nan')), 1)
        else:
            row[f'{stage}_status'] = '—'
            row[f'{stage}_dur_s']  = float('nan')
    rows.append(row)

pd.DataFrame(rows)

## 5. Key findings

Numbers from `accuracy.csv` and `receval_summary.csv` (n = 1 319 for all conditions). ReCEval info scorer: **EleutherAI/pythia-1b**.

| Finding | Observation |
|---|---|
| **Direct FT wins accuracy at base scale** | `student_direct_ft` 5.23 % > `baseline` 4.32 %; matches Ho et al. direct FT (5.08 %). |
| **CoT sets underperform at base scale** | Set A 2.65 %, Set B 3.34 %, Set C 3.03 % — all below baseline. Consistent with Ho et al.: CoT benefit requires scale. |
| **Set C ≈ Set B — calculator patching adds no signal** | The teacher (text-davinci-002) makes arithmetic errors in < 0.1 % of Set B chains. With 91 % of Set C identical to Set B, there is no meaningful extra training signal. |
| **Scale crossover partially confirmed** | Large models improve across the board (`direct_ft_large` 6.90 %, `set_b_large` 5.16 %, `set_c_large` 4.70 %). CoT still trails direct FT at T5-large scale — likely needs more epochs or a larger model. |
| **ReCEval info now differentiated (pythia-1b scorer)** | Direct FT conditions are the only ones with positive info (`direct_ft` +0.24, `direct_ft_large` +0.31). All CoT conditions remain negative (≈ −2.2). Positive info reflects answer-token predictability, not reasoning quality. |
| **Set C shows higher inter-step coherence than Set B** | `student_set_c` inter 0.203 vs `student_set_b` inter 0.061. Calculator-patched chains produce less contradictory intermediate steps — but this does not translate to accuracy gains, confirming the scale threshold hypothesis. |
| **ReCEval intra is not informative** | All conditions score 0.91–0.97. The implementation runs NLI with (step, step) as premise and hypothesis — a sentence trivially entails itself. |
| **Right answer ≠ good reasoning** | Direct FT achieves the highest accuracy and highest info/inter scores. With pythia-1b, this is explained by output brevity — short answer-only outputs are more predictable to the scorer than long CoT chains, regardless of reasoning correctness. |